## Pivot Table Examples

In [1]:
import pandas as pd

In [2]:
## Simple Dataset for Examples 1 and 2

In [3]:
df_simple = pd.DataFrame({
    'Category':    ['A', 'A', 'A', 'B', 'B', 'B'],
    'SubCategory': ['X', 'Y', 'X', 'X', 'Y', 'Y'],
    'Amount':      [ 10,  20,   5,   7,   3,   2]})
df_simple

,Category,SubCategory,Amount
0,A,X,10
1,A,Y,20
2,A,X,5
3,B,X,7
4,B,Y,3
5,B,Y,2


### Example 1
Category (rows) × SubCategory (columns), `sum(Amount)`

In [22]:
pivot1 = pd.pivot_table(df_simple,
    values='Amount',
    index='Category',
    columns='SubCategory',
    aggfunc='sum')

pivot1

SubCategory,X,Y
Category,,
A,15,20
B,7,5


### Example 2
Summarize by Category + SubCategory (row fields), `sum(Amount)`, with Grand Total

In [15]:
pivot2 = pd.pivot_table(df_simple,
    values='Amount',
    index=['Category', 'SubCategory'],
    aggfunc='sum',
    margins=True,
    margins_name='Grand Total')
pivot2_flat = pivot2.reset_index()

pivot2_flat

,Category,SubCategory,Amount
0,A,X,15
1,A,Y,20
2,B,X,7
3,B,Y,5
4,Grand Total,,47


## Example 3 - Sales Dataset
Raw data has three metric columns whose values need to be summarized by `Store` and `ProdType` rows by `Week` + `Year` columns (Final shape is akin to Excel Scenario Model where Scenario columns are keyed by `Week` and `Year`)

In [6]:
df_otb = pd.DataFrame({
    'Store':     ['Store1','Store1','Store1','Store1','Store1','Store1',
                  'Store2','Store2','Store2','Store2','Store2','Store2'],
    'Prodtype':  ['X','Y','Z','X','Y','Z','X','Y','Z','X','Y','Z'],
    'Week':      [1,1,1,2,2,2,1,1,1,2,2,2],
    'Year':      [2025]*12,
    'Discounts': [10,20,30,11,21,31,40,50,60,41,51,61],
    'Markdowns': [100,200,300,101,201,301,400,500,600,401,501,601],
    'COGS':      [1000,2000,3000,1001,2001,3001,4000,5000,6000,4001,5001,6001],
})
df_otb

,Store,Prodtype,Week,Year,Discounts,Markdowns,COGS
0,Store1,X,1,2025,10,100,1000
1,Store1,Y,1,2025,20,200,2000
2,Store1,Z,1,2025,30,300,3000
3,Store1,X,2,2025,11,101,1001
4,Store1,Y,2,2025,21,201,2001
5,Store1,Z,2,2025,31,301,3001
6,Store2,X,1,2025,40,400,4000
7,Store2,Y,1,2025,50,500,5000
8,Store2,Z,1,2025,60,600,6000
9,Store2,X,2,2025,41,401,4001


### Example 3 - Sales Data

Pandas equivalent: melt the three analyte columns to long form, then pivot with `index=["Store","Metric","Prodtype"]`.

In [17]:
# Melt wide analyte columns to long form: Discounts/Markdowns/COGS → Metric, Value
df_otb_long = df_otb.melt(
    id_vars=['Store', 'Prodtype', 'Week', 'Year'],
    value_vars=['Discounts', 'Markdowns', 'COGS'],
    var_name='Metric',
    value_name='Value',
)

# Pivot: Store/Metric/Prodtype rows (Metric at index position 2, matching DataPivotFieldPosition=2)
#        Week+Year columns, Value summed
pivot3 = pd.pivot_table(
    df_otb_long,
    values='Value',
    index=['Store', 'Metric', 'Prodtype'],
    columns=['Week', 'Year'],
    aggfunc='sum',
)

pivot3

Week                          1     2
Year                       2025  2025
Store  Metric    Prodtype            
Store1 COGS      X         1000  1001
                 Y         2000  2001
                 Z         3000  3001
       Discounts X           10    11
                 Y           20    21
                 Z           30    31
       Markdowns X          100   101
                 Y          200   201
                 Z          300   301
Store2 COGS      X         4000  4001
                 Y         5000  5001
                 Z         6000  6001
       Discounts X           40    41
                 Y           50    51
                 Z           60    61
       Markdowns X          400   401
                 Y          500   501
                 Z          600   601

In [8]:
# Flatten column multi-index for display
pivot3.columns = [f'Wk{w}_Yr{y}' for w, y in pivot3.columns]
pivot3_flat = pivot3.reset_index()
pivot3_flat

,Store,Metric,Prodtype,Wk1_Yr2025,Wk2_Yr2025
0,Store1,COGS,X,1000,1001
1,Store1,COGS,Y,2000,2001
2,Store1,COGS,Z,3000,3001
3,Store1,Discounts,X,10,11
4,Store1,Discounts,Y,20,21
5,Store1,Discounts,Z,30,31
6,Store1,Markdowns,X,100,101
7,Store1,Markdowns,Y,200,201
8,Store1,Markdowns,Z,300,301
9,Store2,COGS,X,4000,4001
